# nf-core/scrnaseq — FASTQ → Count Matrix Alignment

**Genesis Workbench** — Single Cell Module

Runs [nf-core/scrnaseq](https://nf-co.re/scrnaseq) on a Databricks cluster to align FASTQ files and produce count matrices (h5ad) for scanpy.

**Prerequisites**: Cluster init scripts `install_nextflow.sh` + `install_sra_tools.sh`
**Triggered by**: Streamlit app → Data Ingestion → Nextflow Alignment

In [0]:
# Widget parameters (set by Databricks Jobs API from Streamlit)
dbutils.widgets.text("fastq_dir", "", "FASTQ Directory")
dbutils.widgets.text("output_dir", "", "Output Directory")
dbutils.widgets.dropdown("aligner", "starsolo", ["starsolo", "salmon", "kallisto", "cellranger"], "Aligner")
dbutils.widgets.dropdown("genome", "GRCh38", ["GRCh38", "GRCm39", "GRCm38"], "Reference Genome")
dbutils.widgets.dropdown("chemistry", "auto", ["auto", "10XV2", "10XV3", "10XV3HT", "10XV5"], "10x Chemistry")
dbutils.widgets.text("sample_name", "", "Sample Name (optional)")
dbutils.widgets.text("pipeline_version", "2.7.1", "nf-core/scrnaseq Version")
dbutils.widgets.text("extra_args", "", "Extra Nextflow Args")

fastq_dir = dbutils.widgets.get("fastq_dir")
output_dir = dbutils.widgets.get("output_dir")
aligner = dbutils.widgets.get("aligner")
genome = dbutils.widgets.get("genome")
chemistry = dbutils.widgets.get("chemistry")
sample_name = dbutils.widgets.get("sample_name") or None
pipeline_version = dbutils.widgets.get("pipeline_version")
extra_args = dbutils.widgets.get("extra_args") or None

print(f"FASTQ dir:  {fastq_dir}")
print(f"Output dir: {output_dir}")
print(f"Aligner:    {aligner}")
print(f"Genome:     {genome}")
print(f"Chemistry:  {chemistry}")

In [0]:
import os, sys

# Validate inputs
assert fastq_dir and os.path.isdir(fastq_dir), f"FASTQ directory not found: {fastq_dir}"
assert output_dir, "Output directory must be specified"

# Verify Nextflow is installed
import subprocess
try:
    nxf_version = subprocess.check_output(["nextflow", "-version"], stderr=subprocess.STDOUT, text=True)
    print("✅ Nextflow available:")
    print(nxf_version.strip())
except FileNotFoundError:
    raise RuntimeError(
        "❌ Nextflow not found. Add init script to cluster: "
        "/Volumes/dhbl_discovery_us_dev/genesis_schema/scanpy_reference/init_scripts/install_nextflow.sh"
    )

# Count FASTQ files
import glob
fastq_count = len(glob.glob(os.path.join(fastq_dir, "**/*.fastq*"), recursive=True))
fq_count = len(glob.glob(os.path.join(fastq_dir, "**/*.fq*"), recursive=True))
print(f"\n📁 Found {fastq_count + fq_count} FASTQ files in {fastq_dir}")

In [0]:
# Add the app utils to path so we can import nextflow_pipeline
sys.path.insert(0, "/Workspace/Users/andrew_forman@eisai.com/.bundle/genesis_workbench/core/prod/files/app")
from utils.nextflow_pipeline import build_samplesheet, ALIGNERS

# Build samplesheet from FASTQ files
samplesheet_path = os.path.join(output_dir, "samplesheet.csv")
sheet_result = build_samplesheet(fastq_dir, samplesheet_path, sample_name)

if "error" in sheet_result:
    raise RuntimeError(f"❌ Samplesheet error: {sheet_result['error']}")

print(f"✅ Samplesheet: {sheet_result['samplesheet_path']}")
print(f"   Samples: {sheet_result['n_samples']}")
print(f"   FASTQ pairs: {sheet_result['n_fastq_pairs']}")
print(f"   Aligner: {ALIGNERS[aligner]['name']} — {ALIGNERS[aligner]['description']}")
print(f"\nSamplesheet contents:")
with open(samplesheet_path) as f:
    print(f.read())

In [0]:
from utils.nextflow_pipeline import build_nextflow_command
import time

# Build and run the Nextflow command
cmd = build_nextflow_command(
    samplesheet_path=samplesheet_path,
    output_dir=output_dir,
    aligner=aligner,
    genome=genome,
    chemistry=chemistry,
    pipeline_version=pipeline_version,
    extra_args=extra_args,
)

print(f"🚀 Running nf-core/scrnaseq v{pipeline_version}")
print(f"   Command: {' '.join(cmd)}")
print(f"   Started: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "="*60 + "\n")

# Run with live output
log_path = os.path.join(output_dir, "nextflow_run.log")
start_time = time.time()

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=output_dir,
)

# Stream output to both notebook and log file
with open(log_path, "w") as log_file:
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

process.wait()
elapsed = time.time() - start_time

print("\n" + "="*60)
print(f"\n{'✅' if process.returncode == 0 else '❌'} Pipeline finished in {elapsed/60:.1f} minutes (exit code: {process.returncode})")

if process.returncode != 0:
    raise RuntimeError(f"Nextflow failed with exit code {process.returncode}. Check log: {log_path}")

In [0]:
from utils.nextflow_pipeline import find_pipeline_outputs

outputs = find_pipeline_outputs(output_dir, aligner)

print("📦 Pipeline Outputs:")
print(f"   Count matrices (filtered): {len(outputs['count_matrices'])}")
for f in outputs['count_matrices'][:5]:
    print(f"     {f}")

print(f"   h5ad files: {len(outputs['h5ad_files'])}")
for f in outputs['h5ad_files'][:5]:
    print(f"     {f}")

if outputs['multiqc_report']:
    print(f"   MultiQC report: {outputs['multiqc_report']}")

print(f"   Raw matrices: {len(outputs['raw_matrices'])}")

In [0]:
from utils.nextflow_pipeline import convert_mtx_to_h5ad
import glob

# Find filtered matrix directories and convert to h5ad
mtx_dirs = set()
for f in outputs['count_matrices']:
    if 'filtered' in f:
        mtx_dirs.add(os.path.dirname(f))

if mtx_dirs and not outputs['h5ad_files']:
    print("🔄 Converting count matrices to h5ad format...")
    for mtx_dir in mtx_dirs:
        sample = os.path.basename(os.path.dirname(mtx_dir)) or "sample"
        h5ad_path = os.path.join(output_dir, f"{sample}_filtered.h5ad")
        result = convert_mtx_to_h5ad(mtx_dir, h5ad_path)
        if result['status'] == 'success':
            print(f"  ✅ {h5ad_path} ({result['n_cells']:,} cells × {result['n_genes']:,} genes)")
            outputs['h5ad_files'].append(h5ad_path)
        else:
            print(f"  ❌ Failed: {result['error']}")

elif outputs['h5ad_files']:
    print("h5ad files already present — skipping conversion")
else:
    print("⚠️  No filtered count matrices found. Check pipeline logs.")

# Set task value for downstream scanpy job
if outputs['h5ad_files']:
    h5ad_path = outputs['h5ad_files'][0]
    dbutils.jobs.taskValues.set(key="h5ad_path", value=h5ad_path)
    print(f"\n📋 Task value set: h5ad_path = {h5ad_path}")
    print(f"\n✅ Ready for scanpy! Copy this path to the Run Analysis tab:")
    print(f"   {h5ad_path}")

In [0]:
# Summary
print("╔══════════════════════════════════════════════════════╗")
print("║  nf-core/scrnaseq Pipeline Complete                  ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Aligner:    {aligner:<40} ║")
print(f"║  Genome:     {genome:<40} ║")
print(f"║  Duration:   {elapsed/60:.1f} minutes{' '*(33-len(f'{elapsed/60:.1f} minutes'))} ║")
print(f"║  h5ad files: {len(outputs['h5ad_files']):<40} ║")
print("╚══════════════════════════════════════════════════════╝")

if outputs['h5ad_files']:
    for f in outputs['h5ad_files']:
        print(f"\n  📄 {f}")